Flask dashboard that works

In [ ]:
# Install dependencies (run this cell in Colab)
!pip install flask pyngrok


In [ ]:
!pip install lime

In [ ]:
!pip install kaleido==0.2.1

In [ ]:
!pip install openai-clip

In [ ]:
!mkdir imgs

In [ ]:
import output_maps
import os

In [ ]:
!pip install dash plotly -q

In [ ]:
import importlib
importlib.reload(output_maps)

In [ ]:
output_maps.precompute()

In [ ]:
!pip show shap

In [ ]:
from flask import Flask, render_template

In [ ]:
import layerwise_maps

In [ ]:
# Create a "templates" folder for Jinja2 HTML
os.makedirs("templates", exist_ok=True)

# example python functions to setup template
def greet_user():
    return "Hello from Python!"

def get_square():
    num = 5
    return f"The square of {num} is {num ** 2}"

def get_random_fact():
    import random
    facts = [
        "Bananas are berries, but strawberries aren’t.",
        "Honey never spoils.",
        "Octopuses have three hearts."
    ]
    return random.choice(facts)


# Flask app pages including sample pages and the output maps/layerwise pages
# Functionality is from both external python files
app = Flask(__name__)

@app.route('/')
def home():
    message = greet_user()
    return render_template("home.html", message=message)

@app.route('/math')
def math_page():
    result = get_square()
    return render_template("math.html", result=result)

@app.route('/fact')
def fact_page():
    fact = get_random_fact()
    return render_template("fact.html", fact=fact)

@app.route('/outputmaps')
def output_maps_page():
    params = ['shap', 'lime', 'gradcam', 'agg']
    items = [output_maps.explain_output(p)[1] for p in params]
    print("TYPE ITEMS 0")
    print(type(items[0]))
    graphJSON = [json.dumps(fig, cls=plotly.utils.PlotlyJSONEncoder) for fig in items]
    return render_template('outputmaps.html', graphJSON=graphJSON)

@app.route('/layerwisemaps')
def layerwise_page():
    # Assuming dash_app is accessible here or passed as an argument if needed
    _ = layerwise_maps.explain_output(dash_app)
    return render_template('layerwisemaps.html')

In [ ]:
import plotly
import plotly.express as px
import json

# @app.route("/testofplotly")
# def testofplotly():
#   # Example Plotly figure
#   df = px.data.iris()
#   fig = px.scatter(df, x='sepal_width', y='sepal_length', color='species',
#                     title='Iris Sepal Dimensions')

#   # Convert figure to JSON
#   graphJSON = json.dumps(fig, cls=plotly.utils.PlotlyJSONEncoder)

#   # Render the template, passing the figure JSON
#   return render_template('testofplotly.html', graphJSON=graphJSON)


In [ ]:
from pyngrok import ngrok
from google.colab import userdata
from flask import Flask, render_template

ngrok.set_auth_token(userdata.get('ngrok'))

dash_app = layerwise_maps.make_dash_app(app, '/dash/')

# Open an ngrok tunnel on the default Flask port 5000
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

# Run Flask
app.run(port=5000)

In [ ]:
#### This cell just saves the external files to a Colab folder in drive for better version control

from google.colab import drive
import os
from datetime import datetime
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create timestamped folder in Drive
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
drive_dir = f'/content/drive/MyDrive/colab_exports/{timestamp}'
os.makedirs(drive_dir, exist_ok=True)

# 3. Copy templates folder
if os.path.exists('templates'):
    shutil.copytree('templates', os.path.join(drive_dir, 'templates'))
else:
    print("No 'templates' folder found in Colab environment.")

# 4. Copy output_maps.py
if os.path.exists('output_maps.py'):
    shutil.copy('output_maps.py', drive_dir)
else:
    print("No 'output_maps.py' file found in Colab environment.")

if os.path.exists('layerwise_maps.py'):
    shutil.copy('layerwise_maps.py', drive_dir)
else:
    print("No 'layerwise_maps.py' file found in Colab environment.")



# 5. List files in the new Drive folder
print(f"Files copied to: {drive_dir}")
!ls -R "$drive_dir"
